# Week 7 – Hugging Face Pipelines

## Introduction
Hugging Face Pipelines provide a simple way to
perform NLP tasks using pre-trained models.

## Objective
To perform sentiment analysis, text summarization,
translation, and question answering using
Hugging Face Pipelines.

In [2]:
from pathlib import Path
import pandas as pd
from transformers import pipeline

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Set dataset folder
DATA = Path("../dataset")

if not DATA.exists():
    DATA = Path("dataset")

# Load input article
input_df = pd.read_csv(DATA / "input_text.csv")

text = str(input_df.iloc[0, 0])

print("Article loaded successfully!")
print(text[:500])

Article loaded successfully!
Artificial Intelligence is changing education. It helps students learn in a personalized way. Students can use AI tools to understand difficult topics, practice questions, and improve their learning. Teachers can also use AI to prepare learning materials and monitor student progress.


In [4]:
# Sentiment Analysis
sentiment_pipeline = pipeline(
    "sentiment-analysis"
)

sentiment_result = sentiment_pipeline(text[:2000])[0]

print("Sentiment:", sentiment_result["label"])
print("Confidence:", sentiment_result["score"])

sentiment_df = pd.DataFrame([{
    "text": text,
    "sentiment": sentiment_result["label"],
    "confidence": sentiment_result["score"]
}])

sentiment_df.to_csv(
    DATA / "sentiment_results.csv",
    index=False
)

print("Sentiment result saved!")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 584.30it/s]


Sentiment: POSITIVE
Confidence: 0.998713493347168
Sentiment result saved!


In [6]:
# Text Summarization

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd
from pathlib import Path

# Load the model
model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Load input article
DATA = Path("../dataset")
if not (DATA / "input_text.csv").exists():
    DATA = Path("dataset")

df = pd.read_csv(DATA / "input_text.csv")
article = str(df.iloc[0, 0])

# Generate summaries
inputs = tokenizer(
    article,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
)

summary_df = []

for length in [60, 100]:
    summary_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=length,
        min_length=20,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    print(f"\nSummary (max_length={length}):")
    print(summary)

    summary_df.append({
        "max_length": length,
        "summary": summary
    })

# Save summaries
pd.DataFrame(summary_df).to_csv(
    DATA / "summarized_text.csv",
    index=False
)

print("\nSummaries saved successfully!")

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 511/511 [00:02<00:00, 207.56it/s]



Summary (max_length=60):
Artificial Intelligence is changing education. It helps students learn in a personalized way. Students can use AI tools to understand difficult topics. Teachers can also use AI to prepare learning materials.

Summary (max_length=100):
Artificial Intelligence is changing education. It helps students learn in a personalized way. Students can use AI tools to understand difficult topics. Teachers can also use AI to prepare learning materials.

Summaries saved successfully!


In [9]:
# Translation using MarianMT

from transformers import MarianMTModel, MarianTokenizer
import pandas as pd

# Load translation model
model_name = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Translate English text to French
text = article

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

translated = model.generate(
    **inputs,
    max_length=512
)

french_text = tokenizer.decode(
    translated[0],
    skip_special_tokens=True
)

print("Original Text:")
print(text)

print("\nFrench Translation:")
print(french_text)

# Save translation
translation_df = pd.DataFrame([{
    "language": "French",
    "original_text": text,
    "translation": french_text
}])

translation_df.to_csv(
    DATA / "translated_text.csv",
    index=False
)

print("\nTranslation saved successfully!")

c:\Users\divya\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 7588.57it/s]


Original Text:
Artificial Intelligence is changing education. It helps students learn in a personalized way. Students can use AI tools to understand difficult topics, practice questions, and improve their learning. Teachers can also use AI to prepare learning materials and monitor student progress.

French Translation:
L'intelligence artificielle est en train de changer l'éducation. Elle aide les étudiants à apprendre de façon personnalisée.Les étudiants peuvent utiliser des outils d'IA pour comprendre des sujets difficiles, pratiquer des questions et améliorer leur apprentissage.

Translation saved successfully!


c:\Users\divya\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\divya\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-fr. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
